In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.12.0.dev20260217+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080


In [5]:
import re
import numpy as np
import napari
from pathlib import Path
from typing import Optional
from scipy.ndimage import zoom as ndi_zoom

from magicgui import magicgui
from magicgui.widgets import Container, TextEdit
from liffile import LifFile
from skimage import util
from skimage.filters import threshold_triangle, median, sobel, gaussian, threshold_yen, apply_hysteresis_threshold
from skimage.measure import label, regionprops_table, regionprops, moments_central
from skimage.morphology import disk, remove_small_objects, remove_small_holes, erosion, closing
from skimage.transform import ProjectiveTransform, warp, probabilistic_hough_line
from scipy.ndimage import rotate, binary_dilation, median_filter
from skimage.draw import line
import tifffile
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.inferers import SlidingWindowInferer, SliceInferer
from monai.networks.nets import UNet
from skimage.metrics import structural_similarity as ssim
import segmentation_models_pytorch as smp
import cupy as cp
from cupyx.scipy import ndimage as cpx_ndimage

In [ ]:
class DeviceSegmentationApp:
    def __init__(
        self,
        low_frac: float = 0.82,
        high_frac: float = 1.10,
        smooth_window: int = 5,
        bin_size: float = 2.0,
        min_run_frac: float = 0.25,
        typical_pct: float = 50.0,
        line_length: int = 400,
        line_gap: int = 900,
        hough_threshold: int = 70,
        mask_sigma: float = 2.0,
        mask_frac_thresh: float = 0.70,
    ):
        self.low_frac = low_frac
        self.high_frac = high_frac
        self.smooth_window = smooth_window
        self.bin_size = bin_size
        self.min_run_frac = min_run_frac
        self.typical_pct = typical_pct
        self.line_length = line_length
        self.line_gap = line_gap
        self.hough_threshold = hough_threshold
        self.mask_sigma = mask_sigma
        self.mask_frac_thresh = mask_frac_thresh

        self.viewer = napari.Viewer()

        self._selected_image_folder: Optional[Path] = None
        self._selected_lif: Optional[Path] = None
        self._image_paths = []
        self._image_choice_map = {}

        self._roi_layer = None
        self._roi_outer_layer = None
        self._active_device_width_um = None
        self._syncing_outer_geometry = False
        self._focus_votes_layer = None
        self._cropped_layer = None
        self._last_image = None
        self._last_image_path = None
        self._last_see_interim_layers = True

        self._last_stack = None
        self._last_focus_downsample = 1
        self._last_focus_n_sampling = 10
        self._last_focus_patch = 50

        self._last_z_step_um = None
        self._last_y_step_um = None
        self._last_x_step_um = None
        self._last_xy_step_um = None
        self._last_center_z0 = None
        self._last_sampled_best_z_min = None
        self._last_sampled_best_z_max = None
        self._last_geometry_best_z_min = None
        self._last_geometry_best_z_max = None
        self._last_geometry_vote_counts = None
        self._last_nz = None
        self._loaded_voxel_um = (None, None, None)

        self._cropped_stack_xy_raw = None
        self._cropped_stack_z_raw = None
        self.cropped_xyz = None
        self._updating_z_widgets = False

        self._z_tracker = {}
        self.images_output = TextEdit(value="")
        try:
            self.images_output.native.setReadOnly(True)
        except Exception:
            pass
        self.images_output.min_height = 120
        self.images_output.max_height = 300

        @magicgui(
            image_source={"label": "Folder/.tif/.lif", "mode": "r"},
            call_button="Load images",
        )
        def list_images(image_source: Path = Path()):
            self._list_images(image_source)

        @magicgui(
            image_choice={"label": "Image", "choices": ["(load images)"], "widget_type": "ComboBox"},
            focus_downsample={"label": "LIF downsample", "min": 1, "max": 16, "step": 1},
            focus_n_sampling={"label": "LIF n_sampling", "min": 4, "max": 100, "step": 1},
            focus_patch={"label": "LIF patch", "min": 5, "max": 512, "step": 1},
            mask_central_region={"label": "Mask central region"},
            see_interim_layers={"label": "See interim layers (debug)"},
            clear_layers={"label": "Clear viewer first"},
            call_button="Segment + View",
        )
        def segment_and_view(
            image_choice: str = "(load images)",
            focus_downsample: int = 4,
            focus_n_sampling: int = 10,
            focus_patch: int = 50,
            mask_central_region: bool = False,
            see_interim_layers: bool = False,
            clear_layers: bool = True,
        ):
            self._segment_and_view(
                image_choice=image_choice,
                focus_downsample=focus_downsample,
                focus_n_sampling=focus_n_sampling,
                focus_patch=focus_patch,
                mask_central_region=mask_central_region,
                see_interim_layers=see_interim_layers,
                clear_layers=clear_layers,
            )

        @magicgui(call_button="Create cropped aligned")
        def apply_crop():
            self._apply_crop_from_roi()

        @magicgui(
            auto_call=True,
            call_button=False,
            device_width_um={"label": "Device width (um)", "min": 0.0, "max": 1000000.0, "step": 1.0},
        )
        def device_width_ok(device_width_um: float = 0.0):
            self._apply_device_width_layer(device_width_um)

        @magicgui(
            z_range_um={"label": "Z range (um)", "min": 1.0, "max": 100000.0, "step": 1.0},
            z_min={"label": "z_min (slice)", "widget_type": "SpinBox", "min": 0, "step": 1},
            z_max={"label": "z_max (slice)", "widget_type": "SpinBox", "min": 0, "step": 1},
            call_button="Crop Z range",
        )
        def crop_z(z_range_um: float = 200.0, z_min: int = 0, z_max: int = 0):
            self._apply_z_crop(z_range_um=z_range_um)

        self.list_images = list_images
        self.segment_and_view = segment_and_view
        self.apply_crop = apply_crop
        self.device_width_ok = device_width_ok
        self.crop_z = crop_z

        self.main_panel = Container(
            widgets=[
                self.list_images,
                self.segment_and_view,
                self.device_width_ok,
                self.apply_crop,
                self.crop_z,
                self.images_output,
            ]
        )
        self.viewer.window.add_dock_widget(self.main_panel, area="right")

        self.segment_and_view.image_choice.changed.connect(self._update_segment_button)
        self.segment_and_view.call_button.enabled = False
        self._update_segment_button()
        self.device_width_ok.device_width_um.enabled = False
        self.crop_z.call_button.enabled = False
        self._set_z_controls_enabled(False, reset_values=True)

        try:
            self.crop_z.z_range_um.changed.connect(lambda *_: self._update_z_crop_bounds(show_warning=True))
            self.crop_z.z_min.changed.connect(lambda *_: self._update_z_range_from_slices(show_warning=True))
            self.crop_z.z_max.changed.connect(lambda *_: self._update_z_range_from_slices(show_warning=True))
        except Exception:
            pass

    # -------- Unified metadata reader (LIF/TIFF) --------
    def _read_voxel_size_um(self, source_path: Optional[Path], source_is_lif: bool, image_index: Optional[int] = None):
        """Return (z_um, y_um, x_um) using one function:
        if lif -> xarray coord steps, elif tif -> TIFF tags.
        """

        def step(axis, xa):
            if axis not in xa.coords or xa.coords[axis].size < 2:
                return None
            return float(xa.coords[axis][1] - xa.coords[axis][0])

        z_um = y_um = x_um = None

        if source_is_lif:
            if self._selected_lif is None:
                return (None, None, None)
            try:
                with LifFile(self._selected_lif) as lif:
                    idx = int(image_index if image_index is not None else 0)
                    if idx < 0 or idx >= len(lif.images):
                        return (None, None, None)
                    img = lif.images[idx]
                    xa = img.asxarray()

                    x_step = step("X", xa)
                    y_step = step("Y", xa)
                    z_step = step("Z", xa)

                    x_um = x_step * 1e6 if x_step is not None else None
                    y_um = y_step * 1e6 if y_step is not None else None
                    z_um = z_step * 1e6 if z_step is not None else None
            except Exception:
                return (None, None, None)
            return (z_um, y_um, x_um)

        elif source_path is not None and str(source_path).lower().endswith((".tif", ".tiff")):
            try:
                with tifffile.TiffFile(str(source_path)) as tif:
                    tif_tags = {}
                    for tag in tif.pages[0].tags.values():
                        tif_tags[tag.name] = tag.value

                    # XY from resolution tags (as requested)
                    if "XResolution" in tif_tags:
                        xres = tif_tags["XResolution"]
                        x_pixel_size_um = 1.0 / (float(xres[0]) / float(xres[1]))
                        x_um = x_pixel_size_um
                    if "YResolution" in tif_tags:
                        yres = tif_tags["YResolution"]
                        y_pixel_size_um = 1.0 / (float(yres[0]) / float(yres[1]))
                        y_um = y_pixel_size_um

                    # Z from IJMetadata first, fallback to ImageDescription spacing
                    try:
                        z_um = float(str(tif_tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
                    except Exception:
                        try:
                            z_um = float(str(tif_tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
                        except Exception:
                            z_um = None
            except Exception:
                return (None, None, None)

            if x_um is None and y_um is not None:
                x_um = y_um
            if y_um is None and x_um is not None:
                y_um = x_um
            return (z_um, y_um, x_um)

        return (None, None, None)

    def _fmt_um(self, v):
        if v is None:
            return "NA"
        try:
            vf = float(v)
        except Exception:
            return "NA"
        if not np.isfinite(vf):
            return "NA"
        return f"{vf:.4g}"

    def _voxel_log_text(self, z_um, y_um, x_um):
        return f"Voxel size (um): x={self._fmt_um(x_um)}, y={self._fmt_um(y_um)}, z={self._fmt_um(z_um)}"

    # -------- Focus helpers (integrated; no separate class) --------
    def _to_gray(self, im):
        if im.ndim == 2:
            return im
        return np.mean(im, axis=-1)

    def _focus_score(self, patch):
        p = np.asarray(self._to_gray(patch), dtype=float)
        return float(np.std(sobel(p)))

    def _curved_plane_refocus(self, stack_zyx, grid=20, patch=50, mask=None):
        Z, H, W = stack_zyx.shape
        ys = np.linspace(patch // 2, H - patch // 2 - 1, grid).astype(int)
        xs = np.linspace(patch // 2, W - patch // 2 - 1, grid).astype(int)

        pts, zs = [], []
        for y in ys:
            for x in xs:
                if mask is not None and not mask[y, x]:
                    continue
                sl = (slice(y - patch // 2, y + patch // 2), slice(x - patch // 2, x + patch // 2))
                f = np.array([self._focus_score(stack_zyx[z][sl]) for z in range(Z)], dtype=np.float32)
                pts.append((x, y))
                zs.append(int(np.argmax(f)))

        if len(zs) < 6:
            scores = []
            for z in range(Z):
                sl = self._to_gray(stack_zyx[z])
                if mask is not None:
                    sl = sl * mask
                scores.append(np.std(sobel(sl)))
            z0 = int(np.argmax(scores))
            img = self._to_gray(stack_zyx[z0])
            zmap = np.full(stack_zyx.shape[1:], z0, dtype=np.int16)
            return img, zmap, np.array(pts), np.array(zs)

        pts = np.array(pts, dtype=np.float32)
        zs = np.array(zs, dtype=np.float32)

        Xc = pts[:, 0] - pts[:, 0].mean()
        Yc = pts[:, 1] - pts[:, 1].mean()
        scale = max(W, H)
        Xn = Xc / scale
        Yn = Yc / scale

        B = np.column_stack((Xn**2, Yn**2, Xn * Yn, Xn, Yn, np.ones_like(Xn)))
        coeffs, *_ = np.linalg.lstsq(B, zs, rcond=None)

        Xg, Yg = np.meshgrid(np.arange(W), np.arange(H))
        mean_x = pts[:, 0].mean()
        mean_y = pts[:, 1].mean()
        Xg_n = (Xg - mean_x) / scale
        Yg_n = (Yg - mean_y) / scale
        zmap = (
            coeffs[0] * Xg_n**2
            + coeffs[1] * Yg_n**2
            + coeffs[2] * Xg_n * Yg_n
            + coeffs[3] * Xg_n
            + coeffs[4] * Yg_n
            + coeffs[5]
        )
        zmap = np.clip(np.rint(zmap).astype(np.int16), 0, Z - 1)
        img = np.take_along_axis(stack_zyx, zmap[None, :, :], axis=0)[0]
        return img, zmap, pts, zs
    def _compute_focus_plane_from_stack(self, stack: Optional[np.ndarray], downsample: int, n_sampling: int, patch: int, source_is_lif: bool = False, image_index: Optional[int] = None):
        # Unified focus path: if LIF, load + normalize here; elif use provided stack.
        if source_is_lif:
            if self._selected_lif is None:
                raise ValueError("No LIF file selected")
            with LifFile(self._selected_lif) as lif:
                idx = int(image_index if image_index is not None else 0)
                img = lif.images[idx]
                image = img.asarray()
            arr = np.asarray(image)
            if arr.ndim == 2:
                arr = arr[np.newaxis, ...]
            elif arr.ndim == 3:
                arr = arr if arr.shape[0] < 64 else arr[np.newaxis, ...]
            elif arr.ndim != 4:
                raise ValueError(f"Unsupported LIF array shape: {arr.shape}")

            self._last_stack = arr.astype(np.float32)
            z_um, y_um, x_um = self._read_voxel_size_um(self._selected_lif, source_is_lif=True, image_index=idx)
            self._last_z_step_um = z_um
            self._last_y_step_um = y_um
            self._last_x_step_um = x_um
            self._last_xy_step_um = np.nanmean([v for v in [x_um, y_um] if v is not None]) if (x_um is not None or y_um is not None) else None
        else:
            arr = np.asarray(stack)

        nz = int(arr.shape[0])
        ds = max(1, int(downsample))
        self._last_focus_downsample = ds

        stack_score_full = np.mean(arr, axis=-1) if arr.ndim == 4 else arr
        focus_full, _, _, zs = self._curved_plane_refocus(stack_score_full, grid=int(n_sampling), patch=int(patch), mask=None)

        if ds > 1:
            focus_out = focus_full[::ds, ::ds]
        else:
            focus_out = focus_full

        self._last_center_z0 = None
        self._last_nz = nz
        if zs is None or len(zs) == 0:
            self._last_sampled_best_z_min = None
            self._last_sampled_best_z_max = None
        else:
            zs_arr = np.asarray(zs, dtype=int)
            zs_arr = zs_arr[(zs_arr >= 0) & (zs_arr < nz)]
            if zs_arr.size > 0:
                counts = np.bincount(zs_arr, minlength=nz)
                self._last_center_z0 = int(np.argmax(counts))
                self._last_sampled_best_z_min = int(zs_arr.min())
                self._last_sampled_best_z_max = int(zs_arr.max())
            else:
                self._last_sampled_best_z_min = None
                self._last_sampled_best_z_max = None

        return focus_out

    # -------- segmentation + geometry --------
    def _is_color_image_2d(self, arr: np.ndarray) -> bool:
        return arr.ndim == 3 and arr.shape[-1] in (3, 4) and arr.shape[0] > 32 and arr.shape[1] > 32

    def _scale_to_uint8_view(self, arr):
        if arr is None:
            return None
        data = np.asarray(arr)
        dmin = float(np.nanmin(data))
        dmax = float(np.nanmax(data))
        if not np.isfinite(dmin) or not np.isfinite(dmax) or dmax <= dmin:
            return np.zeros_like(data, dtype=np.uint8)
        scaled = np.clip((data - dmin) / (dmax - dmin), 0.0, 1.0)
        return (scaled * 255.0).astype(np.uint8)

    def _order_corners_clockwise(self, corners_xy: np.ndarray):
        c = np.asarray(corners_xy, dtype=float)
        centroid = c.mean(axis=0)
        angles = np.arctan2(c[:, 1] - centroid[1], c[:, 0] - centroid[0])
        c = c[np.argsort(angles)]
        start = np.argmin(c[:, 0] + c[:, 1])
        return np.roll(c, -start, axis=0)

    def _crop_rectified_from_corners(self, in_focus_plane: np.ndarray, corners_xy: np.ndarray):
        if corners_xy is None:
            return None
        c = self._order_corners_clockwise(corners_xy)
        w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
        w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
        h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
        h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
        width = int(max(w1, w2))
        height = int(max(h1, h2))
        if width <= 1 or height <= 1:
            return None

        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
        tform = ProjectiveTransform()
        if not tform.estimate(dst, c):
            return None
        warped = warp(in_focus_plane, tform, output_shape=(height, width), order=1, preserve_range=True)
        return warped.astype(in_focus_plane.dtype)

    def _crop_rectified_stack_from_corners(self, stack: np.ndarray, corners_xy: np.ndarray):
        if corners_xy is None or stack is None:
            return None
        c = self._order_corners_clockwise(corners_xy)
        w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
        w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
        h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
        h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
        width = int(max(w1, w2))
        height = int(max(h1, h2))
        if width <= 1 or height <= 1:
            return None

        dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
        tform = ProjectiveTransform()
        if not tform.estimate(dst, c):
            return None

        if stack.ndim == 3:
            warped = np.stack(
                [warp(stack[z], tform, output_shape=(height, width), order=1, preserve_range=True) for z in range(stack.shape[0])],
                axis=0,
            )
        elif stack.ndim == 4:
            warped = np.stack(
                [
                    warp(
                        stack[z],
                        tform,
                        output_shape=(height, width),
                        order=1,
                        preserve_range=True,
                        channel_axis=-1,
                    )
                    for z in range(stack.shape[0])
                ],
                axis=0,
            )
        else:
            return None

        return warped.astype(stack.dtype)

    def _signed_orientation(self, region):
        img = region.image.astype(float)
        mu = moments_central(img)
        angle_rad = 0.5 * np.arctan2(2 * mu[1, 1], mu[2, 0] - mu[0, 2])
        return np.rad2deg(angle_rad)

    def _corners_touch_border(self, corners_xy: np.ndarray, shape, margin=0):
        H, W = shape
        x = corners_xy[:, 0]
        y = corners_xy[:, 1]
        return (x <= margin).any() or (x >= (W - 1 - margin)).any() or (y <= margin).any() or (y >= (H - 1 - margin)).any()

    def _mask_out_organoid(self, in_focus_plane):
        inverted = gaussian(
            util.invert(np.asarray(in_focus_plane, dtype=np.float32)),
            sigma=float(self.mask_sigma),
            mode="nearest",
            preserve_range=True,
        ).astype(np.float32)
        H, W = in_focus_plane.shape[:2]
        xy_area = H * W
        cyi, cxi = H // 2, W // 2
        r = int(min(H, W) * 0.1)
        yy, xx = np.ogrid[:H, :W]
        central_roi = (yy - cyi) ** 2 + (xx - cxi) ** 2 <= r**2

        thresh = threshold_yen(inverted)
        labelled = label(inverted > thresh)
        props = regionprops(labelled)
        if len(props) == 0:
            return np.zeros((H, W), dtype=bool)

        def score(p):
            overlap = np.sum(labelled[central_roi] == p.label)
            py, px = p.centroid
            dist2 = (py - cyi) ** 2 + (px - cxi) ** 2
            return (-overlap, dist2)

        best_prop = min(props, key=score)
        if (best_prop.area > xy_area * self.mask_frac_thresh) or (best_prop.solidity < 0.5):
            thresh = threshold_triangle(inverted)
            labelled = label(inverted > thresh)
            props = regionprops(labelled)
            if len(props) == 0:
                return np.zeros((H, W), dtype=bool)
            best_prop = min(props, key=score)

        organoid_region = labelled == best_prop.label
        organoid_region = remove_small_holes(organoid_region, area_threshold=50000)
        organoid_region = closing(organoid_region, disk(5))
        return organoid_region

    def _oriented_rect_corners_crop_necks_and_flares(self, mask: np.ndarray):
        ys, xs = np.nonzero(mask)
        if xs.size == 0:
            return None, None, None

        cx, cy = xs.mean(), ys.mean()
        centroid_xy = np.array([cx, cy], float)

        mu = moments_central(mask.astype(np.uint8))
        angle_rad = 0.5 * np.arctan2(2 * mu[1, 1], (mu[0, 2] - mu[2, 0]))

        c, s = np.cos(angle_rad), np.sin(angle_rad)
        dx = xs - cx
        dy = ys - cy
        u = c * dx + s * dy
        v = -s * dx + c * dy

        u_span = float(u.max() - u.min())
        v_span = float(v.max() - v.min())
        if v_span >= u_span:
            long, short, long_name = v, u, "v"
        else:
            long, short, long_name = u, v, "u"

        short_min_full = float(short.min())
        short_max_full = float(short.max())
        long_bin = np.floor(long / self.bin_size).astype(int)
        bins, inv = np.unique(long_bin, return_inverse=True)

        short_min = np.full(bins.shape, np.inf)
        short_max = np.full(bins.shape, -np.inf)
        np.minimum.at(short_min, inv, short)
        np.maximum.at(short_max, inv, short)
        widths = short_max - short_min

        if self.smooth_window and self.smooth_window > 1:
            w = int(self.smooth_window)
            if w % 2 == 0:
                w += 1
            pad = w // 2
            widths_s = np.convolve(np.pad(widths, (pad, pad), mode="edge"), np.ones(w) / w, mode="valid")
        else:
            widths_s = widths

        typical_width = float(np.percentile(widths_s, self.typical_pct))
        low_thr = self.low_frac * typical_width
        high_thr = self.high_frac * typical_width
        keep = (~(widths_s < low_thr)) & (~(widths_s > high_thr))
        too_wide = widths_s > high_thr

        best_start = best_end = None
        best_len = 0
        cur_start = cur_end = None
        for i in range(len(bins)):
            if not keep[i]:
                if cur_start is not None:
                    L = cur_end - cur_start + 1
                    if L > best_len:
                        best_len = L
                        best_start, best_end = cur_start, cur_end
                    cur_start = cur_end = None
                continue
            if cur_start is None:
                cur_start = cur_end = bins[i]
            elif bins[i] == cur_end + 1:
                cur_end = bins[i]
            else:
                L = cur_end - cur_start + 1
                if L > best_len:
                    best_len = L
                    best_start, best_end = cur_start, cur_end
                cur_start = cur_end = bins[i]

        if cur_start is not None:
            L = cur_end - cur_start + 1
            if L > best_len:
                best_len = L
                best_start, best_end = cur_start, cur_end

        total_len_bins = int(bins.max() - bins.min() + 1)
        min_run_bins = int(np.ceil(self.min_run_frac * total_len_bins))

        if best_start is None or best_len < min_run_bins:
            long_min = float(long.min())
            long_max = float(long.max())
            crop_width = False
        else:
            long_min = float(best_start * self.bin_size)
            long_max = float((best_end + 1) * self.bin_size)
            band_mask_bins = (bins >= best_start) & (bins <= best_end)
            crop_width = bool(np.any(too_wide & (~band_mask_bins)))

        in_long_band = (long >= long_min) & (long <= long_max)
        if crop_width:
            short_min_use = float(short[in_long_band].min())
            short_max_use = float(short[in_long_band].max())
        else:
            short_min_use = short_min_full
            short_max_use = short_max_full

        if long_name == "v":
            umin, umax = short_min_use, short_max_use
            vmin, vmax = long_min, long_max
        else:
            umin, umax = long_min, long_max
            vmin, vmax = short_min_use, short_max_use

        corners_uv = np.array([[umin, vmin], [umax, vmin], [umax, vmax], [umin, vmax]], dtype=float)
        x = cx + c * corners_uv[:, 0] - s * corners_uv[:, 1]
        y = cy + s * corners_uv[:, 0] + c * corners_uv[:, 1]
        return np.stack([x, y], axis=1), angle_rad, centroid_xy

    def _segment_from_plane(self, in_focus_plane: np.ndarray, mask_central_region: bool, return_debug: bool):
        flag = False
        in_focus_plane = self._to_gray(in_focus_plane)

        median_thresholded = median(np.asarray(in_focus_plane, dtype=np.float32), footprint=disk(7)).astype(np.float32)
        sobel_operated = sobel(median_thresholded).astype(np.float32)
        thresh = threshold_triangle(sobel_operated)
        binary = sobel_operated > thresh

        h, w = in_focus_plane.shape[:2]
        binary[h // 3 : 2 * (h // 3), w // 3 : 2 * (w // 3)] = 0

        organoid_region = None
        if mask_central_region:
            organoid_region = self._mask_out_organoid(in_focus_plane)
            binary[organoid_region] = 0

        labels = label(binary)
        data = regionprops_table(labels, binary, properties=("label", "area", "eccentricity"))
        condition = (data["area"] > 100) & (data["eccentricity"] > 0.5)
        labels_to_dilate = util.map_array(labels, data["label"], data["label"] * condition)

        dilated_output = np.zeros_like(labels, dtype=np.uint8)
        base_selem = np.zeros((31, 31), dtype=bool)
        base_selem[15, :] = 1
        pad = base_selem.shape[0] // 2

        for region in regionprops(labels_to_dilate):
            angle_to_rotate = self._signed_orientation(region)
            rotated_selem = rotate(base_selem.astype(float), angle=90 + angle_to_rotate, reshape=False, order=0) > 0.5

            minr, minc, maxr, maxc = region.bbox
            r0 = max(minr - pad, 0)
            r1 = min(maxr + pad, labels_to_dilate.shape[0])
            c0 = max(minc - pad, 0)
            c1 = min(maxc + pad, labels_to_dilate.shape[1])

            mask_roi = labels_to_dilate[r0:r1, c0:c1] == region.label
            dilated = binary_dilation(mask_roi.astype(bool), structure=rotated_selem.astype(bool))
            dilated_output[r0:r1, c0:c1][dilated] = 255

        post_dilation_mask = np.logical_or(dilated_output, binary)
        clean_labels = label(~post_dilation_mask)
        props = regionprops(clean_labels)
        largest_prop = max(props, key=lambda p: p.area)
        device_mask = clean_labels == largest_prop.label

        edges = reconstructed = reconstructed_mask = None
        new_corners = new_angle_rad = new_centroid_xy = None

        corners, angle_rad, centroid_xy = self._oriented_rect_corners_crop_necks_and_flares(device_mask)
        if corners is None or self._corners_touch_border(corners, device_mask.shape, margin=5):
            flag = True
            edges = remove_small_objects(labels_to_dilate > 0)
            segs = probabilistic_hough_line(
                edges,
                line_length=self.line_length,
                line_gap=self.line_gap,
                threshold=self.hough_threshold,
            )
            reconstructed = np.zeros_like(edges, dtype=bool)
            for (x0, y0), (x1, y1) in segs:
                rr, cc = line(y0, x0, y1, x1)
                reconstructed[rr, cc] = True

            reconstructed_mask = np.logical_or(reconstructed, post_dilation_mask)
            updated_clean_labels = label(~reconstructed_mask)
            props = regionprops(updated_clean_labels)
            largest_prop = max(props, key=lambda p: p.area)
            new_device_mask = updated_clean_labels == largest_prop.label
            new_corners, new_angle_rad, new_centroid_xy = self._oriented_rect_corners_crop_necks_and_flares(new_device_mask)

        if flag:
            final_corners = new_corners
            final_angle_rad = new_angle_rad
            final_centroid_xy = new_centroid_xy
        else:
            final_corners = corners
            final_angle_rad = angle_rad
            final_centroid_xy = centroid_xy

        cropped_rotated = self._crop_rectified_from_corners(in_focus_plane, final_corners)

        if return_debug:
            debug = {
                "median_thresholded": median_thresholded,
                "sobel_operated": sobel_operated,
                "binary": binary,
                "labels_to_dilate": labels_to_dilate,
                "post_dilation_mask": post_dilation_mask,
                "device_mask": device_mask,
                "organoid_region": organoid_region,
                "edges": edges,
                "reconstructed": reconstructed,
                "reconstructed_mask": reconstructed_mask,
                "flag": flag,
                "final_corners": final_corners,
                "final_centroid_xy": final_centroid_xy,
                "final_angle_rad": final_angle_rad,
                "new_corners": new_corners,
                "new_centroid_xy": new_centroid_xy,
                "new_angle_rad": new_angle_rad,
                "cropped_rotated": cropped_rotated,
                "gpu_used_preprocess": False,
                "gpu_used_dilation": False,
            }
            return in_focus_plane, organoid_region, final_corners, cropped_rotated, debug

        return in_focus_plane, organoid_region, final_corners, cropped_rotated

    # -------- IO / GUI flow --------
    def _list_images(self, image_source: Path):
        p = Path(image_source)
        if not p.exists():
            self.images_output.value = "[WARN] Please select a folder, a .tif/.tiff file, or a .lif file."
            return

        self._selected_image_folder = None
        self._selected_lif = None
        self._image_paths = []
        self._last_stack = None
        self._last_focus_downsample = 1
        choices = []
        choice_map = {}

        if p.is_dir():
            self._selected_image_folder = p
            image_files = sorted([q for q in p.iterdir() if q.is_file() and q.suffix.lower() in (".tif", ".tiff")])
            if not image_files:
                self.images_output.value = "[WARN] No .tif/.tiff images found in the selected folder."
                self._image_choice_map = {}
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return
            self._image_paths = image_files
            for i, pth in enumerate(image_files):
                label_txt = f"{i}: {pth.name}"
                choices.append(label_txt)
                choice_map[label_txt] = i
            self._image_choice_map = choice_map
            self.segment_and_view.image_choice.choices = choices
            self.segment_and_view.image_choice.value = choices[0]
            z_um, y_um, x_um = self._read_voxel_size_um(image_files[0], source_is_lif=False)
            self._loaded_voxel_um = (z_um, y_um, x_um)
            self.images_output.value = f"[OK] Found {len(choices)} images in folder. Select one and click Segment + View. {self._voxel_log_text(z_um, y_um, x_um)}"
            self._update_segment_button()
            return

        if p.suffix.lower() == ".lif":
            self._selected_lif = p
            try:
                with LifFile(self._selected_lif) as lif:
                    for i, img in enumerate(lif.images):
                        name = img.name if hasattr(img, "name") and img.name else f"image_{i}"
                        label_txt = f"{i}: {name}"
                        choices.append(label_txt)
                        choice_map[label_txt] = i
            except Exception:
                self.images_output.value = "[ERROR] Unable to read .lif contents."
                self._image_choice_map = {}
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return

            if not choices:
                self.images_output.value = "[WARN] No readable images found inside the selected .lif file."
                self._image_choice_map = {}
                self.segment_and_view.image_choice.choices = ["(load images)"]
                self.segment_and_view.image_choice.value = "(load images)"
                self._update_segment_button()
                return

            self._image_choice_map = choice_map
            self.segment_and_view.image_choice.choices = choices
            self.segment_and_view.image_choice.value = choices[0]
            first_idx = choice_map[choices[0]]
            z_um, y_um, x_um = self._read_voxel_size_um(self._selected_lif, source_is_lif=True, image_index=first_idx)
            self._loaded_voxel_um = (z_um, y_um, x_um)
            self.images_output.value = f"[OK] Found {len(choices)} images in .lif. Select one and click Segment + View. {self._voxel_log_text(z_um, y_um, x_um)}"
            self._update_segment_button()
            return

        if p.suffix.lower() in (".tif", ".tiff"):
            self._image_paths = [p]
            label_txt = f"0: {p.name}"
            self._image_choice_map = {label_txt: 0}
            self.segment_and_view.image_choice.choices = [label_txt]
            self.segment_and_view.image_choice.value = label_txt
            z_um, y_um, x_um = self._read_voxel_size_um(p, source_is_lif=False)
            self._loaded_voxel_um = (z_um, y_um, x_um)
            self.images_output.value = f"[OK] Loaded single .tif file. Select it and click Segment + View. {self._voxel_log_text(z_um, y_um, x_um)}"
            self._update_segment_button()
            return

        self.images_output.value = "[WARN] Unsupported selection. Choose a folder, a .tif/.tiff file, or a .lif file."
        self._image_choice_map = {}
        self.segment_and_view.image_choice.choices = ["(load images)"]
        self.segment_and_view.image_choice.value = "(load images)"
        self._update_segment_button()

    def _segment_and_view(
        self,
        image_choice: str,
        focus_downsample: int,
        focus_n_sampling: int,
        focus_patch: int,
        mask_central_region: bool,
        see_interim_layers: bool,
        clear_layers: bool,
    ):
        source_is_lif = bool(getattr(self, "_selected_lif", None) and self._selected_lif.exists())

        if not source_is_lif and not self._image_paths:
            self.images_output.value = "[WARN] Select a folder/.tif/.lif and click 'Load images' first."
            return
        if not self._image_choice_map:
            self.images_output.value = "[WARN] Click 'Load images' to populate the dropdown."
            return

        image_index = self._image_choice_map.get(image_choice)
        if image_index is None:
            self.images_output.value = "[WARN] Please select an image from the dropdown."
            return

        self._last_focus_n_sampling = int(focus_n_sampling)
        self._last_focus_patch = int(focus_patch)

        try:
            if source_is_lif:
                in_focus_plane = self._compute_focus_plane_from_stack(
                    stack=None,
                    downsample=focus_downsample,
                    n_sampling=focus_n_sampling,
                    patch=focus_patch,
                    source_is_lif=True,
                    image_index=image_index,
                )
                in_focus_plane, organoid_region, final_corners, cropped_rotated, debug = self._segment_from_plane(
                    in_focus_plane,
                    mask_central_region,
                    return_debug=True,
                )
                self._last_image_path = self._selected_lif
            else:
                source_path = self._image_paths[image_index]
                arr = np.asarray(tifffile.imread(str(source_path)))

                z_um, y_um, x_um = self._read_voxel_size_um(Path(source_path), source_is_lif=False)
                self._last_z_step_um = z_um
                self._last_y_step_um = y_um
                self._last_x_step_um = x_um
                self._last_xy_step_um = np.nanmean([v for v in [x_um, y_um] if v is not None]) if (x_um is not None or y_um is not None) else None

                if self._is_color_image_2d(arr) or arr.ndim < 3:
                    in_focus_plane = arr.astype(np.float32)
                    self._last_stack = None
                    self._last_focus_downsample = 1
                    self._last_center_z0 = None
                    self._last_sampled_best_z_min = None
                    self._last_sampled_best_z_max = None
                    self._last_nz = None
                else:
                    stack = arr.astype(np.float32)
                    in_focus_plane = self._compute_focus_plane_from_stack(stack, focus_downsample, focus_n_sampling, focus_patch)
                    self._last_stack = stack
                    self._last_focus_downsample = max(1, int(focus_downsample))

                in_focus_plane, organoid_region, final_corners, cropped_rotated, debug = self._segment_from_plane(
                    in_focus_plane,
                    mask_central_region,
                    return_debug=True,
                )
                self._last_image_path = source_path
        except Exception as e:
            self.images_output.value = f"[ERROR] Segmentation failed: {type(e).__name__}: {e}"
            return

        if clear_layers:
            self.viewer.layers.clear()
            self._roi_layer = None
            self._roi_outer_layer = None
            self._cropped_layer = None

        self._last_image = in_focus_plane
        self._last_see_interim_layers = see_interim_layers
        self._active_device_width_um = 30.0
        self._last_geometry_best_z_min = None
        self._last_geometry_best_z_max = None
        self._last_geometry_vote_counts = None
        self._z_tracker = {}
        if self._last_stack is not None:
            try:
                src_nz = int(np.asarray(self._last_stack).shape[0])
                self._register_z_stage(
                    stage_name="original_stack",
                    nz=src_nz,
                    orig_min=0.0,
                    orig_max=float(max(0, src_nz - 1)),
                    note="Original loaded stack z indices.",
                )
            except Exception:
                self._z_tracker = {}
        if self._roi_outer_layer is not None and self._roi_outer_layer in self.viewer.layers:
            self.viewer.layers.remove(self._roi_outer_layer)
        self._roi_outer_layer = None
        if self._focus_votes_layer is not None and self._focus_votes_layer in self.viewer.layers:
            self.viewer.layers.remove(self._focus_votes_layer)
        self._focus_votes_layer = None
        self._cropped_stack_xy_raw = None
        self._cropped_stack_z_raw = None
        self.cropped_xyz = None
        self.device_width_ok.device_width_um.enabled = True
        self.device_width_ok.device_width_um.value = 30.0
        self.crop_z.call_button.enabled = False
        self._set_z_controls_enabled(False, reset_values=True)

        self._add_layer_if_nonzero(in_focus_plane, name="original", layer_type="image")

        if see_interim_layers:
            self._add_layer_if_nonzero(debug.get("sobel_operated"), name="sobel", layer_type="image")
            self._add_layer_if_nonzero(debug.get("binary").astype(np.uint8), name="binary", layer_type="image")
            self._add_layer_if_nonzero(debug.get("labels_to_dilate").astype(np.int32), name="labels_to_dilate", layer_type="labels")
            final_layer = self._add_layer_if_nonzero(debug.get("post_dilation_mask").astype(np.uint8), name="post_dilation_mask", layer_type="labels")
            if final_layer is not None:
                final_layer.opacity = 0.4
            device_layer = self._add_layer_if_nonzero(debug.get("device_mask").astype(np.uint8), name="device_mask", layer_type="labels")
            if device_layer is not None:
                device_layer.opacity = 0.4
            if organoid_region is not None:
                organoid_layer = self._add_layer_if_nonzero(organoid_region.astype(np.uint8), name="organoid_region", layer_type="labels")
                if organoid_layer is not None:
                    organoid_layer.opacity = 0.4

        force_roi = clear_layers or self._roi_layer is None or len(getattr(self._roi_layer, "data", [])) == 0
        self._set_roi_layer(final_corners, force=force_roi)
        self._update_outer_geometry_from_current_roi(30.0, update_message=False)
        self.images_output.value = (
            "[OK] Segmentation complete. Outer geometry is shown at default Device width=30 um; adjust width as needed."
        )
        self._update_z_crop_bounds(show_warning=False)

    def _set_roi_layer(self, corners_xy, force=False):
        if corners_xy is None:
            self.images_output.value = "[WARN] No corners found for ROI."
            return
        corners_yx = np.asarray(corners_xy)[:, ::-1]
        if self._roi_layer is None or self._roi_layer not in self.viewer.layers:
            self._roi_layer = self.viewer.add_shapes(name="geometry")
            try:
                self._roi_layer.events.data.connect(self._on_roi_layer_data_changed)
            except Exception:
                pass
        if force or len(self._roi_layer.data) == 0:
            self._roi_layer.data = [corners_yx]
        self._roi_layer.shape_type = ["rectangle"]
        self._roi_layer.edge_color = "red"
        self._roi_layer.face_color = "transparent"
        self._roi_layer.editable = True
        self._roi_layer.mode = "select"

    def _on_roi_layer_data_changed(self, event=None):
        if self._syncing_outer_geometry:
            return
        if self._roi_outer_layer is None or self._roi_outer_layer not in self.viewer.layers:
            return

        width_um = self._active_device_width_um
        if width_um is None:
            try:
                width_um = float(self.device_width_ok.device_width_um.value)
            except Exception:
                return

        if not np.isfinite(width_um) or width_um < 0:
            return

        try:
            self._syncing_outer_geometry = True
            self._update_outer_geometry_from_current_roi(width_um, update_message=False)
        finally:
            self._syncing_outer_geometry = False

    def _get_current_roi_corners_xy(self):
        if self._roi_layer is None or len(self._roi_layer.data) == 0:
            return None
        corners_xy = np.asarray(self._roi_layer.data[0])[:, ::-1]
        return self._order_corners_clockwise(corners_xy)

    def _get_current_crop_corners_xy(self):
        if self._roi_outer_layer is not None and self._roi_outer_layer in self.viewer.layers:
            if len(getattr(self._roi_outer_layer, "data", [])) > 0:
                outer_xy = np.asarray(self._roi_outer_layer.data[0])[:, ::-1]
                return self._order_corners_clockwise(outer_xy)
        return self._get_current_roi_corners_xy()

    def _um_to_xy_pixels(self, width_um: float):
        try:
            width_um = float(width_um)
        except Exception:
            return None
        if not np.isfinite(width_um) or width_um < 0:
            return None

        x_um = self._last_x_step_um
        y_um = self._last_y_step_um
        if x_um is None and y_um is None:
            return None
        if x_um is None:
            x_um = y_um
        if y_um is None:
            y_um = x_um

        try:
            x_um = float(x_um)
            y_um = float(y_um)
        except Exception:
            return None
        if x_um <= 0 or y_um <= 0:
            return None

        return width_um / x_um, width_um / y_um

    def _expand_rectangle_corners(self, corners_xy: np.ndarray, expand_x_px: float, expand_y_px: float):
        c = self._order_corners_clockwise(corners_xy)
        center = c.mean(axis=0)

        e0 = c[1] - c[0]
        e1 = c[3] - c[0]
        l0 = float(np.linalg.norm(e0))
        l1 = float(np.linalg.norm(e1))
        if l0 <= 0 or l1 <= 0:
            return None

        u0 = e0 / l0
        u1 = e1 / l1
        h0 = l0 * 0.5
        h1 = l1 * 0.5

        expanded = np.array(
            [
                center - (h0 + expand_x_px) * u0 - (h1 + expand_y_px) * u1,
                center + (h0 + expand_x_px) * u0 - (h1 + expand_y_px) * u1,
                center + (h0 + expand_x_px) * u0 + (h1 + expand_y_px) * u1,
                center - (h0 + expand_x_px) * u0 + (h1 + expand_y_px) * u1,
            ],
            dtype=float,
        )
        return expanded

    def _update_outer_geometry_from_current_roi(self, device_width_um: float, update_message: bool = True):
        corners_xy = self._get_current_roi_corners_xy()
        if corners_xy is None:
            if update_message:
                self.images_output.value = "[WARN] Geometry layer is missing. Run 'Segment + View' again."
            return False

        px_xy = self._um_to_xy_pixels(device_width_um)
        if px_xy is None:
            if update_message:
                self.images_output.value = (
                    "[WARN] Could not convert Device width (um) to pixels. Check image voxel metadata (x/y step)."
                )
            return False

        expand_x_px, expand_y_px = px_xy
        expanded_xy = self._expand_rectangle_corners(corners_xy, expand_x_px, expand_y_px)
        if expanded_xy is None:
            if update_message:
                self.images_output.value = "[WARN] Could not compute expanded geometry."
            return False

        expanded_yx = expanded_xy[:, ::-1]
        if self._roi_outer_layer is None or self._roi_outer_layer not in self.viewer.layers:
            self._roi_outer_layer = self.viewer.add_shapes(name="geometry_device_width")

        self._roi_outer_layer.data = [expanded_yx]
        self._roi_outer_layer.shape_type = ["rectangle"]
        self._roi_outer_layer.edge_color = "yellow"
        self._roi_outer_layer.face_color = "transparent"
        self._roi_outer_layer.editable = False

        if update_message:
            self.images_output.value = (
                f"[OK] Added outer geometry layer using Device width={float(device_width_um):.4g} um "
                f"(~dx={expand_x_px:.2f}px, dy={expand_y_px:.2f}px on each side)."
            )
        return True

    def _apply_device_width_layer(self, device_width_um: float):
        corners_xy = self._get_current_roi_corners_xy()
        if corners_xy is None:
            self.images_output.value = "[WARN] Draw or adjust rectangle first."
            return

        try:
            self._active_device_width_um = float(device_width_um)
        except Exception:
            self.images_output.value = "[WARN] Device width must be numeric."
            return

        self._update_outer_geometry_from_current_roi(self._active_device_width_um, update_message=True)

    def _compute_focus_patch_votes_for_stack(self, stack: np.ndarray, n_sampling: int, patch: int):
        if stack is None:
            return None, None

        stack_arr = np.asarray(stack)
        if stack_arr.ndim == 4:
            stack_gray = np.mean(stack_arr, axis=-1)
        elif stack_arr.ndim == 3:
            stack_gray = stack_arr
        else:
            return None, None

        Z, H, W = stack_gray.shape
        if Z <= 0 or H <= 0 or W <= 0:
            return None, None

        patch = max(5, int(patch))
        grid = max(4, int(n_sampling))
        half = patch // 2

        vote_volume = np.zeros((Z, H, W), dtype=np.float32)

        if H <= patch or W <= patch:
            scores = [np.std(sobel(self._to_gray(stack_gray[z]))) for z in range(Z)]
            z_best = int(np.argmax(scores))
            vote_volume[z_best, H // 2, W // 2] = 1.0
            counts = np.zeros(Z, dtype=np.int32)
            counts[z_best] = 1
            return counts, vote_volume

        ys = np.linspace(half, H - half - 1, grid).astype(int)
        xs = np.linspace(half, W - half - 1, grid).astype(int)

        counts = np.zeros(Z, dtype=np.int32)
        for y in ys:
            for x in xs:
                y0 = max(0, y - half)
                y1 = min(H, y + half)
                x0 = max(0, x - half)
                x1 = min(W, x + half)
                f = np.array([self._focus_score(stack_gray[z, y0:y1, x0:x1]) for z in range(Z)], dtype=np.float32)
                best_z = int(np.argmax(f))
                counts[best_z] += 1
                yy0 = max(0, y - 1)
                yy1 = min(H, y + 2)
                xx0 = max(0, x - 1)
                xx1 = min(W, x + 2)
                vote_volume[best_z, yy0:yy1, xx0:xx1] += 1.0

        return counts, vote_volume

    def _format_geometry_vote_planes(self):
        counts = self._last_geometry_vote_counts
        if counts is None:
            return "none"
        counts = np.asarray(counts).astype(int)
        if counts.size == 0:
            return "none"
        hits = np.flatnonzero(counts > 0)
        if hits.size == 0:
            return "none"
        return ", ".join([f"z{int(i)}:{int(counts[i])}" for i in hits])

    def _geometry_vote_summary_suffix(self):
        txt = self._format_geometry_vote_planes()
        return f" Geometry vote planes (all nonzero): {txt}."

    def _z_linear_interp(self, z_idx: float, in_nz: int, out_min: float, out_max: float):
        if in_nz <= 1:
            return float(out_min)
        return float(out_min + (float(z_idx) / float(in_nz - 1)) * (float(out_max) - float(out_min)))

    def _register_z_stage(self, stage_name: str, nz: int, orig_min: float, orig_max: float, note: str = ""):
        self._z_tracker[str(stage_name)] = {
            "nz": int(max(0, int(nz))),
            "orig_min": float(orig_min),
            "orig_max": float(orig_max),
            "note": str(note),
        }

    def _register_z_stage_from_parent(
        self,
        parent_stage: str,
        stage_name: str,
        child_nz: int,
        parent_z_min: int = 0,
        parent_z_max: Optional[int] = None,
        note: str = "",
    ):
        p = self._z_tracker.get(str(parent_stage))
        if p is None:
            return False

        p_nz = int(p.get("nz", 0))
        if p_nz <= 0:
            return False

        pmin = int(np.clip(int(parent_z_min), 0, max(0, p_nz - 1)))
        pmax = int(p_nz - 1 if parent_z_max is None else np.clip(int(parent_z_max), 0, max(0, p_nz - 1)))
        if pmin > pmax:
            pmin, pmax = pmax, pmin

        o0 = self._z_linear_interp(pmin, p_nz, float(p["orig_min"]), float(p["orig_max"]))
        o1 = self._z_linear_interp(pmax, p_nz, float(p["orig_min"]), float(p["orig_max"]))
        self._register_z_stage(stage_name, int(child_nz), float(o0), float(o1), note=note)
        return True

    def z_to_original(self, stage_name: str, z_index: int):
        stage = self._z_tracker.get(str(stage_name))
        if stage is None:
            return None
        nz = int(stage.get("nz", 0))
        if nz <= 0:
            return None
        zi = int(np.clip(int(z_index), 0, max(0, nz - 1)))
        return self._z_linear_interp(zi, nz, float(stage["orig_min"]), float(stage["orig_max"]))

    def get_z_tracker(self):
        return dict(self._z_tracker)

    def z_tracker_summary(self):
        if not self._z_tracker:
            return "No z-tracker entries yet."
        lines = ["Z tracker (stage -> original z-space)"]
        for key, item in self._z_tracker.items():
            lines.append(
                f"  {key}: nz={int(item['nz'])}, original~[{float(item['orig_min']):.2f}, {float(item['orig_max']):.2f}]"
            )
        return "\n".join(lines)

    def _apply_crop_from_roi(self):
        corners_xy = self._get_current_crop_corners_xy()
        if corners_xy is None:
            self.images_output.value = "[WARN] Draw or adjust geometry first."
            return
        if self._last_image is None:
            self.images_output.value = "[WARN] Run 'Segment + View' first."
            return

        if self._last_stack is not None:
            scale = max(1, int(self._last_focus_downsample))
            cropped_stack = self._crop_rectified_stack_from_corners(self._last_stack, corners_xy * float(scale))
            if cropped_stack is None:
                self.images_output.value = "[WARN] Crop failed for selected geometry (stack)."
                return

            roi_inner_xy = self._get_current_roi_corners_xy()
            roi_stack_only = None
            if roi_inner_xy is not None:
                roi_stack_only = self._crop_rectified_stack_from_corners(self._last_stack, roi_inner_xy * float(scale))
            self._last_geometry_best_z_min = None
            self._last_geometry_best_z_max = None
            self._last_geometry_vote_counts = None

            self._cropped_stack_xy_raw = cropped_stack
            self._cropped_stack_z_raw = None
            self.cropped_xyz = None
            self._register_z_stage_from_parent(
                parent_stage="original_stack",
                stage_name="cropped_xy_raw",
                child_nz=int(self._cropped_stack_xy_raw.shape[0]),
                parent_z_min=0,
                parent_z_max=int(max(0, self._cropped_stack_xy_raw.shape[0] - 1)),
                note="XY-rectified stack from geometry; z preserved from original stack.",
            )
            self.crop_z.call_button.enabled = True
            self._set_z_controls_enabled(True)
            self._sync_z_slice_widget_limits()

            self._cropped_layer = self._add_or_update_image_layer(
                self._cropped_layer,
                self._scale_to_uint8_view(cropped_stack),
                "cropped_rotated",
            )

            self._last_center_z0 = None
            counts, vote_volume = (None, None)
            if roi_stack_only is not None:
                counts, vote_volume = self._compute_focus_patch_votes_for_stack(
                    roi_stack_only.astype(np.float32),
                    n_sampling=int(self._last_focus_n_sampling or 10),
                    patch=int(self._last_focus_patch or 50),
                )

            if counts is not None and len(counts) > 0:
                self._last_center_z0 = int(np.argmax(counts))
                self._last_nz = int(len(counts))
                self._last_geometry_vote_counts = np.asarray(counts, dtype=int)
                nz_hits = np.flatnonzero(np.asarray(counts) > 0)
                if nz_hits.size > 0:
                    self._last_sampled_best_z_min = int(nz_hits.min())
                    self._last_sampled_best_z_max = int(nz_hits.max())
                    self._last_geometry_best_z_min = int(nz_hits.min())
                    self._last_geometry_best_z_max = int(nz_hits.max())
                else:
                    self._last_sampled_best_z_min = None
                    self._last_sampled_best_z_max = None
                    self._last_geometry_best_z_min = None
                    self._last_geometry_best_z_max = None

                if vote_volume is not None:
                    self._focus_votes_layer = self._add_or_update_image_layer(
                        self._focus_votes_layer,
                        self._scale_to_uint8_view(vote_volume),
                        "focus_votes_geometry_only",
                    )

                top_n = min(5, len(counts))
                top_idx = np.argsort(counts)[::-1][:top_n]
                top_txt = ", ".join([f"z{int(i)}:{int(counts[i])}" for i in top_idx])
                all_txt = self._format_geometry_vote_planes()
                self.images_output.value = (
                    f"[OK] Cropped aligned stack created from current geometry. center_z={self._last_center_z0}. "
                    f"Geometry-only focus patch votes (inner geometry only; top planes): {top_txt}. "
                    f"All nonzero voted planes: {all_txt}. Now choose Z range and click Crop Z range."
                )
            else:
                self.images_output.value = (
                    f"[OK] Cropped aligned stack created from current geometry. center_z={self._last_center_z0}. "
                    "Now choose Z range and click Crop Z range."
                )

            self._update_z_crop_bounds(show_warning=False)
            return

        cropped_img = self._crop_rectified_from_corners(self._last_image, corners_xy)
        if cropped_img is None:
            self.images_output.value = "[WARN] Crop failed for selected geometry."
            return
        self.images_output.value = "[OK] Cropped aligned image created from current geometry."

    # -------- z-crop --------
    def _get_active_stack_for_z(self):
        return self._cropped_stack_xy_raw if self._cropped_stack_xy_raw is not None else self._last_stack

    def _set_z_controls_enabled(self, enabled: bool, reset_values: bool = False):
        try:
            self.crop_z.z_range_um.enabled = bool(enabled)
            self.crop_z.z_min.enabled = bool(enabled)
            self.crop_z.z_max.enabled = bool(enabled)
        except Exception:
            pass

        if reset_values:
            try:
                self._updating_z_widgets = True
                self.crop_z.z_min.value = 0
                self.crop_z.z_max.value = 0
            except Exception:
                pass
            finally:
                self._updating_z_widgets = False

    def _sync_z_slice_widget_limits(self):
        try:
            nz = int(self._cropped_stack_xy_raw.shape[0]) if self._cropped_stack_xy_raw is not None else 0
            max_z = max(0, nz - 1)
            self.crop_z.z_min.min = 0
            self.crop_z.z_max.min = 0
            self.crop_z.z_min.max = max_z
            self.crop_z.z_max.max = max_z
        except Exception:
            pass

    def _compute_z_min_max_slices(self, z_range_um: float):
        try:
            z_range_um = float(z_range_um)
        except Exception:
            return None, None

        stack = self._get_active_stack_for_z()
        if stack is None or self._last_center_z0 is None or self._last_z_step_um is None or float(self._last_z_step_um) <= 0:
            return None, None

        nz = int(stack.shape[0])
        self._last_nz = nz
        z_range_px = max(1, int(round(z_range_um / float(self._last_z_step_um))))
        if z_range_px >= nz:
            return 0, nz - 1

        half = z_range_px // 2
        z0 = int(np.clip(int(self._last_center_z0), 0, nz - 1))
        lower = z0 - half
        upper = lower + (z_range_px - 1)

        if lower < 0:
            return 0, z_range_px - 1
        if upper > (nz - 1):
            return nz - z_range_px, nz - 1
        return int(lower), int(upper)

    def _update_z_crop_bounds(self, *_, show_warning: bool = False):
        if self._updating_z_widgets or self._cropped_stack_xy_raw is None:
            return
        try:
            z_range_um = float(self.crop_z.z_range_um.value)
        except Exception:
            return

        z_min, z_max = self._compute_z_min_max_slices(z_range_um)
        if z_min is None or z_max is None:
            return

        try:
            self._updating_z_widgets = True
            nz = int(self._cropped_stack_xy_raw.shape[0])
            self.crop_z.z_min.max = max(0, nz - 1)
            self.crop_z.z_max.max = max(0, nz - 1)
            self.crop_z.z_min.value = int(z_min)
            self.crop_z.z_max.value = int(z_max)
        finally:
            self._updating_z_widgets = False

        if show_warning and self._last_geometry_best_z_min is not None:
            if (self._last_geometry_best_z_min < z_min) or (self._last_geometry_best_z_max > z_max):
                self.images_output.value = (
                    f"[WARN] Z-range {z_range_um}um -> slices {z_min}..{z_max} "
                    f"does NOT cover geometry sampled best-zs ({self._last_geometry_best_z_min}..{self._last_geometry_best_z_max})."
                    f"{self._geometry_vote_summary_suffix()}"
                )

    def _update_z_range_from_slices(self, *_, show_warning: bool = False):
        if self._updating_z_widgets or self._cropped_stack_xy_raw is None:
            return
        nz = int(self._cropped_stack_xy_raw.shape[0])
        try:
            z_min = int(self.crop_z.z_min.value)
            z_max = int(self.crop_z.z_max.value)
        except Exception:
            return

        z_min = int(np.clip(z_min, 0, nz - 1))
        z_max = int(np.clip(z_max, 0, nz - 1))
        if z_min > z_max:
            z_min, z_max = z_max, z_min

        try:
            self._updating_z_widgets = True
            self.crop_z.z_min.value = z_min
            self.crop_z.z_max.value = z_max
        finally:
            self._updating_z_widgets = False

        if self._last_z_step_um is not None and float(self._last_z_step_um) > 0:
            try:
                self._updating_z_widgets = True
                self.crop_z.z_range_um.value = float((z_max - z_min + 1) * float(self._last_z_step_um))
            finally:
                self._updating_z_widgets = False

        if show_warning and self._last_geometry_best_z_min is not None:
            if (self._last_geometry_best_z_min < z_min) or (self._last_geometry_best_z_max > z_max):
                self.images_output.value = (
                    f"[WARN] slices {z_min}..{z_max} do NOT cover geometry sampled best-zs "
                    f"({self._last_geometry_best_z_min}..{self._last_geometry_best_z_max})."
                    f"{self._geometry_vote_summary_suffix()}"
                )

    def _apply_z_crop(self, z_range_um: float = 200.0):
        if self._cropped_stack_xy_raw is None:
            self.images_output.value = "[WARN] Create cropped aligned stack first."
            return

        try:
            z_min = int(self.crop_z.z_min.value)
            z_max = int(self.crop_z.z_max.value)
        except Exception:
            z_min = z_max = None

        nz = int(self._cropped_stack_xy_raw.shape[0])
        if z_min is None or z_max is None:
            z_min, z_max = self._compute_z_min_max_slices(z_range_um)
        if z_min is None or z_max is None:
            self.images_output.value = "[WARN] Cannot determine z_min/z_max."
            return

        z_min = int(np.clip(z_min, 0, nz - 1))
        z_max = int(np.clip(z_max, 0, nz - 1))
        if z_min > z_max:
            z_min, z_max = z_max, z_min

        self._cropped_stack_z_raw = self._cropped_stack_xy_raw[z_min : z_max + 1]
        self.cropped_xyz = self._cropped_stack_z_raw
        self._register_z_stage_from_parent(
            parent_stage="cropped_xy_raw",
            stage_name="cropped_xyz",
            child_nz=int(self._cropped_stack_z_raw.shape[0]),
            parent_z_min=int(z_min),
            parent_z_max=int(z_max),
            note="User-selected z crop from geometry-rectified stack.",
        )
        self._cropped_layer = self._add_or_update_image_layer(
            self._cropped_layer,
            self._scale_to_uint8_view(self._cropped_stack_z_raw),
            "cropped_rotated",
        )

        mapped_min = self.z_to_original("cropped_xyz", 0)
        mapped_max = self.z_to_original("cropped_xyz", int(max(0, self._cropped_stack_z_raw.shape[0] - 1)))
        if mapped_min is None or mapped_max is None:
            self.images_output.value = f"[OK] Z-cropped stack created (slices {z_min}..{z_max})."
        else:
            self.images_output.value = (
                f"[OK] Z-cropped stack created (slices {z_min}..{z_max}). "
                f"Original-z approx range: [{mapped_min:.2f}, {mapped_max:.2f}] from source stack."
            )

    def get_cropped_outputs(self):
        """Return (cropped_xyz, None) for backward compatibility."""
        return self.cropped_xyz, None

    # -------- viewer helpers --------
    def _add_or_update_image_layer(self, layer, data, name):
        if layer is not None and layer in self.viewer.layers:
            layer.data = data
            return layer
        return self._add_layer_if_nonzero(data, name=name, layer_type="image")

    def _has_signal(self, arr) -> bool:
        return arr is not None and np.any(arr)

    def _add_layer_if_nonzero(self, data, name, layer_type="image", **kwargs):
        if not self._has_signal(data):
            return None
        if layer_type == "labels":
            return self.viewer.add_labels(data, name=name, **kwargs)
        return self.viewer.add_image(data, name=name, **kwargs)

    def _update_segment_button(self, *_):
        image_value = self.segment_and_view.image_choice.value
        image_ok = bool(self._image_choice_map) and image_value in self._image_choice_map
        self.segment_and_view.call_button.enabled = image_ok

app = DeviceSegmentationApp()
viewer = app.viewer
segment_and_view = app.segment_and_view
list_images = app.list_images
images_output = app.images_output
main_panel = app.main_panel
get_cropped_outputs = app.get_cropped_outputs
get_z_tracker = app.get_z_tracker

def map_stage_z_to_original(stage_name: str, z_index: int):
    return app.z_to_original(stage_name, z_index)

def print_z_tracker():
    print(app.z_tracker_summary())


cropped_xyz = app.cropped_xyz
print("cropped_xyz:", None if cropped_xyz is None else cropped_xyz.shape)
def pix2pix_translation(stack_input: np.ndarray, model_path: str, device: str = "cuda", n_iter: int = 1) -> np.ndarray:
    class LegacyGenerator(nn.Module):
        def __init__(self, dropout_p: float = 0.4):
            super().__init__()
            self.unet = UNet(
                spatial_dims=3,
                in_channels=1,
                out_channels=1,
                channels=(32, 64, 128, 256, 512),
                strides=(1, 2, 2, 2, 1),
                num_res_units=3,
                dropout=float(dropout_p),
            )

        def forward(self, x):
            return F.relu(self.unet(x))

    checkpoint = torch.load(model_path, map_location="cpu")
    state_dict = checkpoint.get("state_dict", checkpoint)
    gen_items = [(k, v) for k, v in state_dict.items() if k.startswith("generator.")]
    if gen_items:
        gen_sd = {k[len("generator."):]: v for k, v in gen_items}
    elif "generator" in state_dict and isinstance(state_dict["generator"], dict):
        gen_sd = state_dict["generator"]
    else:
        raise RuntimeError("Could not find generator weights with prefix 'generator.'.")
    gen_sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in gen_sd.items()}

    hp = checkpoint.get("hyper_parameters", {}) if isinstance(checkpoint, dict) else {}
    dropout_p = float(hp.get("generator_dropout_p", 0.4)) if isinstance(hp, dict) else 0.4
    model = LegacyGenerator(dropout_p=dropout_p)
    model.load_state_dict(gen_sd, strict=True)
    model = model.to(device).eval()

    stack = np.asarray(stack_input, dtype=np.float32)
    stack = (stack - stack.min()) / (stack.max() - stack.min())
    input_tensor = torch.tensor(stack[None, None, ...], device="cpu").float()

    inferer = SlidingWindowInferer(
        roi_size=(32, 512, 512),
        overlap=(0.75, 0.25, 0.25),
        sw_batch_size=1,
        sw_device=device,
        device="cpu",
        progress=True,
    )

    with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
        preds = torch.stack([
            inferer(inputs=input_tensor, network=model).to("cpu")
            for _ in range(max(1, int(n_iter)))
        ])
        preds = torch.clip(preds, 0, 1)
        preds = torch.mean(preds, dim=0)

    out = preds.squeeze().cpu().numpy().astype(np.float32)
    model.to("cpu")
    del input_tensor, preds
    torch.cuda.empty_cache()
    gc.collect()
    return out


def image_resizer(
    stack_input: np.ndarray,
    src_voxel_um: tuple[float, float, float],
    ref_voxel_um: tuple[float, float, float],
    order: int = 3,
    use_gpu: bool = True,
) -> np.ndarray:
    stack = np.asarray(stack_input, dtype=np.float32)
    src = np.array(src_voxel_um, dtype=float)
    ref = np.array(ref_voxel_um, dtype=float)
    old_shape = np.array(stack.shape[:3], dtype=float)
    new_shape = np.maximum(1, np.rint(old_shape * (src / ref))).astype(int)
    zoom_factors = tuple(float(n) / float(o) for n, o in zip(new_shape, old_shape))

    if all(np.isclose(zf, 1.0) for zf in zoom_factors):
        return stack.astype(np.float32, copy=False)

    if use_gpu:
        out = cpx_ndimage.zoom(cp.asarray(stack), zoom=zoom_factors, order=order).get()
    else:
        out = ndi_zoom(stack, zoom=zoom_factors, order=order, prefilter=False)
    return out.astype(np.float32)


def u_net_segmentation(stack_input: np.ndarray, model_path: str, device: str = "cuda") -> tuple[np.ndarray, np.ndarray]:
    def _adapt_input_conv(in_chans, conv_weight):
        conv_type = conv_weight.dtype
        conv_weight = conv_weight.float()
        o_ch, i_ch, k_h, k_w = conv_weight.shape
        if in_chans == 1:
            if i_ch > 3:
                conv_weight = conv_weight.reshape(o_ch, i_ch // 3, 3, k_h, k_w)
                conv_weight = conv_weight.sum(dim=2, keepdim=False)
            else:
                conv_weight = conv_weight.sum(dim=1, keepdim=True)
        elif in_chans != 3:
            repeat = int(np.ceil(in_chans / 3))
            conv_weight = conv_weight.repeat(1, repeat, 1, 1)[:, :in_chans, :, :]
            conv_weight *= (3 / float(in_chans))
        return conv_weight.to(conv_type)

    model = smp.Unet(encoder_name="mit_b5", classes=1, in_channels=3, encoder_weights=None)
    if hasattr(model, "encoder") and hasattr(model.encoder, "patch_embed1"):
        new_weights = _adapt_input_conv(in_chans=1, conv_weight=model.encoder.patch_embed1.proj.weight)
        model.encoder.patch_embed1.proj = torch.nn.Conv2d(
            in_channels=1, out_channels=64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3)
)
        with torch.no_grad():
            model.encoder.patch_embed1.proj.weight = torch.nn.parameter.Parameter(new_weights)

    checkpoint = torch.load(model_path, map_location="cpu")
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
        model.load_state_dict({(k[6:] if k.startswith("model.") else k): v for k, v in state_dict.items()})
    else:
        model.load_state_dict(checkpoint)
    model = model.to(device).eval()

    tensor = torch.tensor(np.asarray(stack_input, dtype=np.float32)[None, None, ...], device="cpu").float()
    tensor_shape = tensor.shape

    axial_inferer = SliceInferer(
        roi_size=(1024, 1024), sw_batch_size=4, progress=True, mode="gaussian",
        overlap=0.5, device="cpu", sw_device=device
    )
    with torch.no_grad():
        proba_axial = torch.sigmoid(axial_inferer(tensor, model)).squeeze().to("cpu").numpy()

    if tensor_shape[2] < 256:
        pad_d = 256 - tensor_shape[2]
        pad_pre = pad_d // 2
        pad_post = pad_d - pad_pre
        tensor = F.pad(tensor, (0, 0, 0, 0, pad_pre, pad_post), mode="constant", value=-1)

    coronal_inferer = SliceInferer(
        roi_size=(256, 256), sw_batch_size=32, spatial_dim=1, progress=True, mode="gaussian",
        overlap=0.5, device="cpu", sw_device=device
    )
    sagittal_inferer = SliceInferer(
        roi_size=(256, 256), sw_batch_size=32, spatial_dim=2, progress=True, mode="gaussian",
        overlap=0.5, device="cpu", sw_device=device
    )
    with torch.no_grad():
        proba_cor = torch.sigmoid(coronal_inferer(tensor, model)).squeeze().to("cpu").numpy()
        proba_sag = torch.sigmoid(sagittal_inferer(tensor, model)).squeeze().to("cpu").numpy()

    if tensor_shape[2] < 256:
        pad_d = 256 - tensor_shape[2]
        pad_pre = pad_d // 2
        proba_cor = proba_cor[pad_pre:pad_pre + tensor_shape[2], :, :]
        proba_sag = proba_sag[pad_pre:pad_pre + tensor_shape[2], :, :]

    prob_map = np.mean(np.array([proba_axial, proba_cor, proba_sag]), axis=0)

    filt = cpx_ndimage.median_filter(cp.asarray(prob_map), size=7).get()
    seg_mask = apply_hysteresis_threshold(filt, 0.2, 0.5).astype(np.uint8)

    model.to("cpu")
    del tensor
    torch.cuda.empty_cache()
    return prob_map.astype(np.float32), seg_mask
pix2pix_ckpt = Path("luca_models") / "epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
seg_ckpt = Path("luca_models") / "best_full.pth"

ref_voxel_um_legacy = (5.0, 2.0, 2.0)
iso_voxel_um_legacy = (2.0, 2.0, 2.0)

if app.cropped_xyz is None:
    print("[INFO] No cropped stack yet. Run 'Segment + View' then 'Create cropped aligned' first.")
else:
    stack_input_raw = np.asarray(app.cropped_xyz, dtype=np.float32)
    lz, ly, lx = getattr(app, "_loaded_voxel_um", (None, None, None))
    z_um = lz if lz is not None else float(getattr(app, "_last_z_step_um"))
    y_um = ly if ly is not None else float(getattr(app, "_last_y_step_um") or getattr(app, "_last_xy_step_um"))
    x_um = lx if lx is not None else float(getattr(app, "_last_x_step_um") or getattr(app, "_last_xy_step_um"))

    stack_input = image_resizer(
        stack_input=stack_input_raw,
        src_voxel_um=(z_um, y_um, x_um),
        ref_voxel_um=ref_voxel_um_legacy,
        order=3,
        use_gpu=True,
    )

    vessel_pred_from_cropped = pix2pix_translation(
        stack_input=stack_input,
        model_path=str(pix2pix_ckpt),
        device="cuda",
        n_iter=1,
    )

    vessel_pred_iso = image_resizer(
        stack_input=vessel_pred_from_cropped,
        src_voxel_um=ref_voxel_um_legacy,
        ref_voxel_um=iso_voxel_um_legacy,
        order=3,
        use_gpu=True,
    )

    vessel_proba, vessel_mask = u_net_segmentation(
        stack_input=vessel_pred_iso,
        model_path=str(seg_ckpt),
        device="cuda",
    )

    print("Legacy spacing path:")
    print("  raw voxel (z,y,x) um:", (z_um, y_um, x_um))
    print("  pix2pix input voxel um:", ref_voxel_um_legacy, "shape:", stack_input.shape)
    print("  segmentation input voxel um:", iso_voxel_um_legacy, "shape:", vessel_pred_iso.shape)
    target_voxel_um = tuple(float(v) for v in iso_voxel_um_legacy)

    original_iso = image_resizer(
        stack_input=stack_input_raw,
        src_voxel_um=(float(z_um), float(y_um), float(x_um)),
        ref_voxel_um=target_voxel_um,
        order=1,
        use_gpu=True,
    )

    translation_iso = image_resizer(
        stack_input=vessel_pred_from_cropped,
        src_voxel_um=ref_voxel_um_legacy,
        ref_voxel_um=target_voxel_um,
        order=1,
        use_gpu=True,
    )

    prob_iso = image_resizer(
        stack_input=vessel_proba,
        src_voxel_um=iso_voxel_um_legacy,
        ref_voxel_um=target_voxel_um,
        order=1,
        use_gpu=True,
    )

    seg_iso = image_resizer(
        stack_input=vessel_mask.astype(np.float32),
        src_voxel_um=iso_voxel_um_legacy,
        ref_voxel_um=target_voxel_um,
        order=0,
        use_gpu=True,
    ).astype(np.uint8)

    viewer = napari.Viewer()
    viewer.add_image(original_iso, name="original", scale=target_voxel_um)
    viewer.add_image(translation_iso, name="translation", scale=target_voxel_um)
    viewer.add_image(prob_iso, name="pred_mask", scale=target_voxel_um)
    viewer.add_labels(seg_iso, name="segmentation", scale=target_voxel_um)
    print("DeviceSegmentationApp validation OK")
    print("Displayed layers at shared voxel size (z,y,x) um:", target_voxel_um)
    print("original:", original_iso.shape)
    print("translation:", translation_iso.shape)
    print("pred_mask:", prob_iso.shape)
    print("segmentation:", seg_iso.shape)



cropped_xyz: None
[INFO] No cropped stack yet. Run 'Segment + View' then 'Create cropped aligned' first.


In [7]:
required_symbols = [
    "image_resizer",
    "pix2pix_translation",
    "u_net_segmentation",
    "pix2pix_ckpt",
    "seg_ckpt",
]
missing = [name for name in required_symbols if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required symbols: {missing}. Run the model/function definition cell first.")

if "z_tracker" not in globals():
    z_tracker = {}

def _zt_register(stage_name: str, nz: int, orig_min: float, orig_max: float, note: str = ""):
    z_tracker[str(stage_name)] = {
        "nz": int(max(0, int(nz))),
        "orig_min": float(orig_min),
        "orig_max": float(orig_max),
        "note": str(note),
    }

def _zt_linear(z_idx: float, in_nz: int, out_min: float, out_max: float):
    if int(in_nz) <= 1:
        return float(out_min)
    return float(out_min + (float(z_idx) / float(in_nz - 1)) * (float(out_max) - float(out_min)))

def _zt_register_from_parent(parent_stage: str, stage_name: str, child_nz: int, parent_z_min: int = 0, parent_z_max: int = None, note: str = ""):
    p = z_tracker.get(str(parent_stage))
    if p is None:
        return False
    p_nz = int(p.get("nz", 0))
    if p_nz <= 0:
        return False
    pmin = int(np.clip(int(parent_z_min), 0, max(0, p_nz - 1)))
    pmax = int(p_nz - 1 if parent_z_max is None else np.clip(int(parent_z_max), 0, max(0, p_nz - 1)))
    if pmin > pmax:
        pmin, pmax = pmax, pmin
    o0 = _zt_linear(pmin, p_nz, float(p["orig_min"]), float(p["orig_max"]))
    o1 = _zt_linear(pmax, p_nz, float(p["orig_min"]), float(p["orig_max"]))
    _zt_register(stage_name, int(child_nz), float(o0), float(o1), note=note)
    return True

def map_stage_z_to_original(stage_name: str, z_index: int):
    stage = z_tracker.get(str(stage_name))
    if stage is None:
        return None
    nz = int(stage.get("nz", 0))
    if nz <= 0:
        return None
    zi = int(np.clip(int(z_index), 0, max(0, nz - 1)))
    return _zt_linear(zi, nz, float(stage["orig_min"]), float(stage["orig_max"]))

if hasattr(app, "get_z_tracker"):
    try:
        z_tracker.update(dict(app.get_z_tracker()))
    except Exception:
        pass

stack_input_raw = np.asarray(app.cropped_xyz, dtype=np.float32)
lz, ly, lx = getattr(app, "_loaded_voxel_um", (None, None, None))
z_um = lz if lz is not None else float(getattr(app, "_last_z_step_um"))
y_um = ly if ly is not None else float(getattr(app, "_last_y_step_um") or getattr(app, "_last_xy_step_um"))
x_um = lx if lx is not None else float(getattr(app, "_last_x_step_um") or getattr(app, "_last_xy_step_um"))

ref_voxel_um_legacy = (5.0, 2.0, 2.0)
iso_voxel_um_legacy = (2.0, 2.0, 2.0)

stack_input = image_resizer(
    stack_input=stack_input_raw,
    src_voxel_um=(z_um, y_um, x_um),
    ref_voxel_um=ref_voxel_um_legacy,
    order=3,
    use_gpu=True,
)


In [8]:

vessel_pred_from_cropped = pix2pix_translation(
    stack_input=stack_input,
    model_path=str(pix2pix_ckpt),
    device="cuda",
    n_iter=1,
)


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\monai\networks\nets\unet.py:130: UserWarning: `len(strides) > len(channels) - 1`, the last 1 values of strides will not be used.
  warnings.warn(f"`len(strides) > len(channels) - 1`, the last {delta} values of strides will not be used.")
  0%|          | 0/48 [00:00<?, ?it/s]c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\monai\inferers\utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\c

In [9]:
vessel_pred_iso = image_resizer(
    stack_input=vessel_pred_from_cropped,
    src_voxel_um=ref_voxel_um_legacy,
    ref_voxel_um=iso_voxel_um_legacy,
    order=3,
    use_gpu=True,
)

parent_stage = "cropped_xyz" if "cropped_xyz" in z_tracker else ("cropped_xy_raw" if "cropped_xy_raw" in z_tracker else "original_stack")
_zt_register_from_parent(parent_stage, "stack_input", int(np.asarray(stack_input).shape[0]), 0, int(max(0, np.asarray(stack_input).shape[0] - 1)), note="Resampled cropped input for pix2pix.")
_zt_register_from_parent("stack_input", "vessel_pred_from_cropped", int(np.asarray(vessel_pred_from_cropped).shape[0]), 0, int(max(0, np.asarray(vessel_pred_from_cropped).shape[0] - 1)), note="Pix2pix output from cropped input.")
_zt_register_from_parent("vessel_pred_from_cropped", "vessel_pred_iso", int(np.asarray(vessel_pred_iso).shape[0]), 0, int(max(0, np.asarray(vessel_pred_iso).shape[0] - 1)), note="Resampled pix2pix output for U-Net input.")

True

In [10]:
# Segmentation + structured outputs (xy-cropped and xyz-cropped)
required_symbols = [
    "u_net_segmentation",
    "image_resizer",
    "ndi_zoom",
    "app",
    "vessel_pred_iso",
    "vessel_pred_from_cropped",
    "z_um",
    "y_um",
    "x_um",
    "iso_voxel_um_legacy",
    "ref_voxel_um_legacy",
]
missing = [name for name in required_symbols if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required symbols: {missing}. Run the previous translation prep cell first.")

if app.cropped_xyz is None:
    raise RuntimeError("No cropped stack found. Run 'Segment + View', 'Create cropped aligned', and (optionally) 'Crop Z range' first.")

def _resample_to_shape(arr: np.ndarray, target_shape: tuple[int, int, int], order: int) -> np.ndarray:
    arr = np.asarray(arr)
    if tuple(arr.shape[:3]) == tuple(target_shape):
        return arr
    zoom_factors = (
        float(target_shape[0]) / float(arr.shape[0]),
        float(target_shape[1]) / float(arr.shape[1]),
        float(target_shape[2]) / float(arr.shape[2]),
    )
    return ndi_zoom(arr, zoom=zoom_factors, order=order, prefilter=False)

def _crop_xy_equal_margin(arr: np.ndarray, crop_y_px: int, crop_x_px: int) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim < 3:
        return arr
    y0 = int(max(0, crop_y_px))
    x0 = int(max(0, crop_x_px))
    y1 = int(arr.shape[1] - y0)
    x1 = int(arr.shape[2] - x0)
    if y1 <= y0 or x1 <= x0:
        raise RuntimeError(f"XY trim too large for shape {arr.shape}: crop_y={crop_y_px}, crop_x={crop_x_px}.")
    slicer = [slice(None)] * arr.ndim
    slicer[1] = slice(y0, y1)
    slicer[2] = slice(x0, x1)
    return arr[tuple(slicer)]

def _pad_xy_equal_margin(arr: np.ndarray, pad_y_px: int, pad_x_px: int, target_shape: tuple[int, int, int], fill_value: float = 0.0) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim < 3:
        return arr
    pad_spec = [(0, 0)] * arr.ndim
    pad_spec[1] = (int(max(0, pad_y_px)), int(max(0, pad_y_px)))
    pad_spec[2] = (int(max(0, pad_x_px)), int(max(0, pad_x_px)))
    padded = np.pad(arr, pad_spec, mode="constant", constant_values=fill_value)
    if tuple(padded.shape[:3]) != tuple(target_shape):
        padded = _resample_to_shape(padded, target_shape, order=0 if np.issubdtype(padded.dtype, np.integer) else 1)
    return padded

vessel_proba, vessel_mask = u_net_segmentation(
    stack_input=vessel_pred_iso,
    model_path=str(seg_ckpt),
    device="cuda",
)

display_voxel_um = tuple(float(v) for v in iso_voxel_um_legacy)

original_iso_uncropped_xy = image_resizer(
    stack_input=np.asarray(app.cropped_xyz, dtype=np.float32),
    src_voxel_um=(float(z_um), float(y_um), float(x_um)),
    ref_voxel_um=display_voxel_um,
    order=1,
    use_gpu=True,
)

translation_iso_uncropped_xy = image_resizer(
    stack_input=np.asarray(vessel_pred_from_cropped, dtype=np.float32),
    src_voxel_um=ref_voxel_um_legacy,
    ref_voxel_um=display_voxel_um,
    order=1,
    use_gpu=True,
)

segmentation_mask_uncropped_xy = image_resizer(
    stack_input=np.asarray(vessel_mask, dtype=np.float32),
    src_voxel_um=iso_voxel_um_legacy,
    ref_voxel_um=display_voxel_um,
    order=0,
    use_gpu=True,
).astype(np.uint8)

target_shape_zyx = tuple(int(v) for v in segmentation_mask_uncropped_xy.shape[:3])
original_iso_uncropped_xy = _resample_to_shape(original_iso_uncropped_xy, target_shape_zyx, order=1).astype(np.float32)
translation_iso_uncropped_xy = _resample_to_shape(translation_iso_uncropped_xy, target_shape_zyx, order=1).astype(np.float32)
segmentation_mask_uncropped_xy = _resample_to_shape(segmentation_mask_uncropped_xy, target_shape_zyx, order=0).astype(np.uint8)

width_um = float(getattr(app, "_active_device_width_um", 30.0))
_, y_um_disp, x_um_disp = (float(v) for v in display_voxel_um)
crop_y = int(np.rint(width_um / y_um_disp))
crop_x = int(np.rint(width_um / x_um_disp))

translation_iso_xy_cropped = _crop_xy_equal_margin(translation_iso_uncropped_xy, crop_y, crop_x).astype(np.float32)
segmentation_mask_xy_cropped = _crop_xy_equal_margin(segmentation_mask_uncropped_xy, crop_y, crop_x).astype(np.uint8)

translation_iso_xy = _pad_xy_equal_margin(translation_iso_xy_cropped, crop_y, crop_x, target_shape_zyx, fill_value=0.0).astype(np.float32)
segmentation_mask_xy = _pad_xy_equal_margin(segmentation_mask_xy_cropped, crop_y, crop_x, target_shape_zyx, fill_value=0).astype(np.uint8)

  0%|          | 0/170 [00:00<?, ?it/s]c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\monai\inferers\utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
  6%|▋         | 52/816 [00:14<03:30,  3.64it/s] 


KeyboardInterrupt: 

In [36]:
gmin=2
gmax=13
# gmin = getattr(app, "_last_geometry_best_z_min", None)
# gmax = getattr(app, "_last_geometry_best_z_max", None)
# print(gmin, gmax)
# gmin = 8
# gmax = 24
try:
    selected_z_min = int(app.crop_z.z_min.value)
    selected_z_max = int(app.crop_z.z_max.value)
except Exception:
    selected_z_min = None
    selected_z_max = None

seg_nz = int(segmentation_mask_xy.shape[0])
raw_nz = int(np.asarray(app.cropped_xyz).shape[0])

if gmin is None or gmax is None or selected_z_min is None or selected_z_max is None:
    seg_z_min = 0
    seg_z_max = max(0, seg_nz - 1)
else:
    local_min = int(gmin) - int(selected_z_min)
    local_max = int(gmax) - int(selected_z_min)
    local_min, local_max = sorted((local_min, local_max))
    local_min_clip = int(np.clip(local_min, 0, max(0, raw_nz - 1)))
    local_max_clip = int(np.clip(local_max, 0, max(0, raw_nz - 1)))
    if raw_nz > 1 and seg_nz > 1:
        seg_z_min = int(np.rint(local_min_clip * (seg_nz - 1) / (raw_nz - 1)))
        seg_z_max = int(np.rint(local_max_clip * (seg_nz - 1) / (raw_nz - 1)))
    else:
        seg_z_min = 0
        seg_z_max = max(0, seg_nz - 1)
    seg_z_min = int(np.clip(seg_z_min, 0, max(0, seg_nz - 1)))
    seg_z_max = int(np.clip(seg_z_max, 0, max(0, seg_nz - 1)))
    if seg_z_min > seg_z_max:
        seg_z_min, seg_z_max = seg_z_max, seg_z_min

segmentation_mask_xyz = np.asarray(segmentation_mask_xy[seg_z_min:seg_z_max + 1], dtype=np.uint8)

# Keep aliases for compatibility with older cells
original_iso_aligned = original_iso_uncropped_xy
translation_iso_aligned = translation_iso_xy
segmentation_mask_aligned = segmentation_mask_xy
vessel_mask_bestz = segmentation_mask_xyz

if "z_tracker" in globals():
    _zt_register_from_parent("vessel_pred_iso", "vessel_mask", int(np.asarray(vessel_mask).shape[0]), 0, int(max(0, np.asarray(vessel_mask).shape[0] - 1)), note="U-Net binary mask output.")
    _zt_register_from_parent("vessel_mask", "segmentation_mask_uncropped_xy", int(np.asarray(segmentation_mask_uncropped_xy).shape[0]), 0, int(max(0, np.asarray(segmentation_mask_uncropped_xy).shape[0] - 1)), note="Mask on display voxel grid before XY trim.")
    _zt_register_from_parent("segmentation_mask_uncropped_xy", "segmentation_mask_xy", int(np.asarray(segmentation_mask_xy).shape[0]), 0, int(max(0, np.asarray(segmentation_mask_xy).shape[0] - 1)), note=f"XY trim by device width {width_um:.4g} um per side.")
    _zt_register_from_parent("segmentation_mask_xy", "segmentation_mask_xyz", int(np.asarray(segmentation_mask_xyz).shape[0]), int(seg_z_min), int(seg_z_max), note="Best-z XYZ crop from XY-trimmed segmentation.")

print("Segmentation postprocessing complete")
print("  original_iso_uncropped_xy:", tuple(int(v) for v in np.asarray(original_iso_uncropped_xy).shape[:3]))
print("  translation_iso_xy:", tuple(int(v) for v in np.asarray(translation_iso_xy).shape[:3]))
print("  segmentation_mask_xy:", tuple(int(v) for v in np.asarray(segmentation_mask_xy).shape[:3]))
print("  segmentation_mask_xyz:", tuple(int(v) for v in np.asarray(segmentation_mask_xyz).shape[:3]))

Segmentation postprocessing complete
  original_iso_uncropped_xy: (145, 3481, 1668)
  translation_iso_xy: (145, 3481, 1668)
  segmentation_mask_xy: (145, 3481, 1668)
  segmentation_mask_xyz: (69, 3481, 1668)


In [38]:
# Single final viewer
required_symbols = [
    "napari",
    "display_voxel_um",
    "original_iso_uncropped_xy",
    "translation_iso_xy",
    "segmentation_mask_xy",
    "segmentation_mask_xyz",
]
missing = [name for name in required_symbols if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required symbols: {missing}. Run the previous two cells first.")

viewer = napari.Viewer()
viewer.add_image(original_iso_uncropped_xy, name="original_cropped_image_uncropped_xy", scale=display_voxel_um)
viewer.add_image(translation_iso_xy, name="image_translation_xy", scale=display_voxel_um)
viewer.add_labels(segmentation_mask_xy, name="segmentation_mask_cropped_xy", scale=display_voxel_um)
viewer.add_labels(segmentation_mask_xyz, name="segmentation_mask_cropped_xyz", scale=display_voxel_um)

print("Final viewer opened")
print("  original_cropped_image_uncropped_xy:", tuple(int(v) for v in np.asarray(original_iso_uncropped_xy).shape[:3]))
print("  image_translation_xy:", tuple(int(v) for v in np.asarray(translation_iso_xy).shape[:3]))
print("  segmentation_mask_cropped_xy:", tuple(int(v) for v in np.asarray(segmentation_mask_xy).shape[:3]))
print("  segmentation_mask_cropped_xyz:", tuple(int(v) for v in np.asarray(segmentation_mask_xyz).shape[:3]))

Final viewer opened
  original_cropped_image_uncropped_xy: (145, 3481, 1668)
  image_translation_xy: (145, 3481, 1668)
  segmentation_mask_cropped_xy: (145, 3481, 1668)
  segmentation_mask_cropped_xyz: (69, 3481, 1668)
